In [26]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [27]:
df = pd.read_csv(r"C:\데크_기말고사\backend\saramin_cluster_limited.csv")

print(df.shape)
df.head()


(160, 7)


,cluster,search_keyword,job_title,company,region,date,url
0,cluster_frontend,프론트엔드 개발자,경력 웹프론트엔드 개발자채용 (CMS/React/JavaScript),(주)솔로몬텍,서울 서초구 경력2년↑ 초대졸↑ 정규직,채용시,https://www.saramin.co.kr/zf_user/jobs/relay/v...
1,cluster_frontend,프론트엔드 개발자,프론트엔드 개발자모집,(주)솔텍시스템,서울 강서구 경력무관 초대졸↑ 정규직,상시채용,https://www.saramin.co.kr/zf_user/jobs/relay/v...
2,cluster_frontend,프론트엔드 개발자,프론트엔드(javascript/typescript/react/vue)신입개발자 채용,㈜엠투소프트,서울 성동구 신입 초대졸↑ 정규직,~ 12/28(일),https://www.saramin.co.kr/zf_user/jobs/relay/v...
3,cluster_frontend,프론트엔드 개발자,백엔드(spring) / 프론트엔드(react/vue) 개발자 모집,(주)아이스크림미디어,경기 성남시 분당구 경력 8~15년 대졸↑ 정규직·프리랜서,~ 01/25(일),https://www.saramin.co.kr/zf_user/jobs/relay/v...
4,cluster_frontend,프론트엔드 개발자,네이블 Backend 개발자 및 Frontend 개발자 채용,(주)네이블,경기 성남시 분당구 경력3년↑ 대졸↑ 정규직,~ 01/11(일),https://www.saramin.co.kr/zf_user/jobs/relay/v...


In [28]:
# 예시: recommender 결과에서 받은 값
user_cluster_key = "cluster_system"

user_cluster_key



'cluster_system'

In [29]:
df_cluster = df[df["cluster"] == user_cluster_key].reset_index(drop=True)

print("클러스터 내 채용 공고 수:", len(df_cluster))
df_cluster.head()


클러스터 내 채용 공고 수: 30


,cluster,search_keyword,job_title,company,region,date,url
0,cluster_system,시스템 엔지니어,시스템엔지니어 신입 채용,(주)엘림넷,서울 금천구 신입 학력무관 정규직,~ 12/20(토),https://www.saramin.co.kr/zf_user/jobs/relay/v...
1,cluster_system,시스템 엔지니어,운영지원2팀시스템 엔지니어(신입 및 경력 5년이하) 모집,(주)사이버다임,"서울 송파구 신입·경력 초대졸↑ 정규직 3,684 만원",~ 01/31(토),https://www.saramin.co.kr/zf_user/jobs/relay/v...
2,cluster_system,시스템 엔지니어,(주)도연시스템즈시스템 엔지니어/신입,(주)도연시스템즈,경기 성남시 분당구 경력무관 초대졸↑ 정규직,~ 12/23(화),https://www.saramin.co.kr/zf_user/jobs/relay/v...
3,cluster_system,시스템 엔지니어,[이테크시스템] 클라우드 보안 솔루션(SASE) 엔지니어 모집,(주)이테크시스템,서울 중구 경력무관 초대졸↑ 정규직,~ 12/31(수),https://www.saramin.co.kr/zf_user/jobs/relay/v...
4,cluster_system,시스템 엔지니어,[이테크시스템] LINUX 엔지니어 경력 모집,(주)이테크시스템,서울 중구 경력무관 초대졸↑ 정규직,~ 12/24(수),https://www.saramin.co.kr/zf_user/jobs/relay/v...


In [30]:
cluster_job_text = " ".join(
    df_cluster["search_keyword"].dropna().unique().tolist()
)

cluster_job_text


'시스템 엔지니어 네트워크 엔지니어 임베디드 개발자'

In [31]:
df_cluster["text"] = (
    df_cluster["job_title"].fillna("") + " " +
    df_cluster["search_keyword"].fillna("")
)

df_cluster[["job_title", "text"]].head()


,job_title,text
0,시스템엔지니어 신입 채용,시스템엔지니어 신입 채용 시스템 엔지니어
1,운영지원2팀시스템 엔지니어(신입 및 경력 5년이하) 모집,운영지원2팀시스템 엔지니어(신입 및 경력 5년이하) 모집 시스템 엔지니어
2,(주)도연시스템즈시스템 엔지니어/신입,(주)도연시스템즈시스템 엔지니어/신입 시스템 엔지니어
3,[이테크시스템] 클라우드 보안 솔루션(SASE) 엔지니어 모집,[이테크시스템] 클라우드 보안 솔루션(SASE) 엔지니어 모집 시스템 엔지니어
4,[이테크시스템] LINUX 엔지니어 경력 모집,[이테크시스템] LINUX 엔지니어 경력 모집 시스템 엔지니어


In [32]:
corpus = [cluster_job_text] + df_cluster["text"].tolist()

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=2000
)

X = vectorizer.fit_transform(corpus)

cluster_vec = X[0]
job_vecs = X[1:]


In [33]:
similarities = cosine_similarity(cluster_vec, job_vecs)[0]

df_cluster["similarity"] = similarities

df_cluster[["job_title", "similarity"]].sort_values(
    by="similarity", ascending=False
).head(10)


,job_title,similarity
14,사물인터넷(IoT)네트워크 엔지니어,0.420787
5,시스템 엔지니어(경력),0.363178
9,[이테크시스템] Cisco 네트워크 엔지니어 모집,0.316761
13,[서울]네트워크 엔지니어(경력),0.300287
29,임베디드응용보안도구개발자(5↑),0.248738
27,임베디드 소프트웨어 개발자 (Junior),0.247629
23,임베디드시스템 S/W 개발자 모집,0.228585
2,(주)도연시스템즈시스템 엔지니어/신입,0.225837
11,[기술지원센터] 네트워크 및 보안부문 엔지니어 채용,0.219847
21,임베디드 및 장치 어플리케이션 S/W 개발자 채용,0.218982


In [34]:
TOP_N = 5

top10_jobs = (
    df_cluster.sort_values(by="similarity", ascending=False)
    .head(TOP_N)
)

top10_jobs.shape


(5, 9)

In [35]:
final_output = top10_jobs[[
    "job_title",
    "company",
    "region",
    "url",
    "similarity"
]]

final_output


,job_title,company,region,url,similarity
14,사물인터넷(IoT)네트워크 엔지니어,㈜어빌리티시스템즈,서울 송파구 신입·경력 학력무관 정규직,https://www.saramin.co.kr/zf_user/jobs/relay/v...,0.420787
5,시스템 엔지니어(경력),(주)카카오,경기 성남시 분당구 경력 1~4년 학력무관 정규직,https://www.saramin.co.kr/zf_user/jobs/relay/v...,0.363178
9,[이테크시스템] Cisco 네트워크 엔지니어 모집,(주)이테크시스템,서울 중구 경력2년↑ 초대졸↑ 정규직,https://www.saramin.co.kr/zf_user/jobs/relay/v...,0.316761
13,[서울]네트워크 엔지니어(경력),에스넷시스템(주),서울 강남구 경력7년↑ 초대졸↑ 정규직,https://www.saramin.co.kr/zf_user/jobs/relay/v...,0.300287
29,임베디드응용보안도구개발자(5↑),쿤텍(주),경기 성남시 경력 5~10년 초대졸↑ 정규직,https://www.saramin.co.kr/zf_user/jobs/relay/v...,0.248738


In [36]:
final_output = final_output.copy()

final_output["reason"] = (
    "사용자에게 추천된 직무 클러스터와 연관성이 높은 채용 공고입니다."
)



In [37]:
import os
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. 사람인 CSV 불러오기
BASE = r"C:\데크_기말고사\backend"
csv_path = os.path.join(BASE, "saramin_cluster_limited.csv")

df = pd.read_csv(csv_path)

print("CSV 로드:", df.shape)
print(df.columns)

# 2. 공고 텍스트 만들기
# (검색어 + 직무명 → 의미 최대화)
df["job_text"] = (
    df["search_keyword"].fillna("") + " " + df["job_title"].fillna("")
).str.strip()

# 3. TF-IDF 벡터화
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["job_text"])

print("TF-IDF 완료:", X.shape)

# 4. pkl 파일 저장
with open(os.path.join(BASE, "saramin_vectorizer.pkl"), "wb") as f:
    pickle.dump(vectorizer, f)

with open(os.path.join(BASE, "saramin_tfidf.pkl"), "wb") as f:
    pickle.dump(X, f)

print("✅ pkl 저장 완료")


CSV 로드: (160, 7)
Index(['cluster', 'search_keyword', 'job_title', 'company', 'region', 'date',
       'url'],
      dtype='object')
TF-IDF 완료: (160, 876)
✅ pkl 저장 완료
